# Practical 2 (Simplified) — Solving the 8-Puzzle with Heuristic Search

### CSE276 · Artificial Intelligence Foundations · Unit II

This is the **easier version** of Practical 2. It uses the same ideas as the full notebook but with
shorter code, one new idea per cell, and small outputs you can actually read.

**What you will build, in order:**

1. A board, and a way to print it
2. A function that lists the legal moves
3. A **heuristic** — a guess at how far the goal is
4. **Hill Climbing** — the simplest informed search
5. **Best First Search** — keeps alternatives, uses `h` only
6. **A\*** — keeps alternatives, uses `g + h`

> **How to use this notebook:** run the cells from top to bottom with Shift+Enter.
> Do not skip a cell — later cells use the functions defined earlier.

## 1. The board

A board is a **tuple of 9 numbers**, read left to right, top to bottom. `0` is the blank space.

```text
   START              GOAL

  1  3  6           1  2  3
  5  _  2    -->    4  5  6
  4  7  8           7  8  _
```

In [ ]:
# The goal we always aim for
GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)

# A board that is 8 moves away from the goal
START = (1, 3, 6,
         5, 0, 2,
         4, 7, 8)

# A very easy board, only 2 moves away - we use it to test our code
EASY = (1, 2, 3,
        4, 5, 6,
        0, 7, 8)


def show(board):
    """Print a board as 3 rows."""
    for row in range(3):
        three = board[row * 3: row * 3 + 3]
        print(" ".join(str(n) if n != 0 else "_" for n in three))


print("START:")
show(START)
print()
print("GOAL:")
show(GOAL)

## 2. What moves are possible?

Only the **blank** moves. It can slide Up, Down, Left or Right — as long as it stays on the board.

```text
   blank in the middle        blank in a corner
   -> 4 possible moves        -> 2 possible moves

      1  3  6                    1  3  6
      5 [_] 2                    5  2  8
      4  7  8                    4  7 [_]
```

The function below returns a **list of new boards**, one for each legal slide.

In [ ]:
def get_moves(board):
    """Return a list of the boards we can reach in one slide."""
    result = []

    blank = board.index(0)          # where the blank is (0 to 8)
    row = blank // 3                # which row  (0, 1 or 2)
    col = blank % 3                 # which column

    # Up, Down, Left, Right
    for row_change, col_change in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        new_row = row + row_change
        new_col = col + col_change

        # Only keep the move if it stays on the board
        if 0 <= new_row <= 2 and 0 <= new_col <= 2:
            target = new_row * 3 + new_col

            new_board = list(board)                     # copy
            new_board[blank] = new_board[target]        # tile slides into the blank
            new_board[target] = 0                       # blank moves to where the tile was
            result.append(tuple(new_board))

    return result


print("From START we can reach", len(get_moves(START)), "boards:")
print()
for b in get_moves(START):
    show(b)
    print()

## 3. A heuristic — guessing how far the goal is

Uninformed search (BFS, DFS in Practical 1) has no idea whether it is getting closer. A **heuristic**
`h(board)` estimates how many moves are still needed.

- `h = 0` means we are **at** the goal.
- A **small** h means the board looks close to the goal.
- A **large** h means it looks far away.

The simplest heuristic: **count the tiles that are not in the right place.** We do not count the blank.

> A heuristic is only a *guess*. It must never be **too big** — a heuristic that never overestimates is
> called **admissible**, and that is what makes A\* find the shortest answer.

In [ ]:
def misplaced(board):
    """How many tiles are NOT in their goal position? (the blank does not count)"""
    wrong = 0
    for i in range(9):
        if board[i] != 0 and board[i] != GOAL[i]:
            wrong = wrong + 1
    return wrong


print("h(GOAL)  =", misplaced(GOAL), " <- zero, because we are already there")
print("h(EASY)  =", misplaced(EASY))
print("h(START) =", misplaced(START))

## 4. A better heuristic — Manhattan distance

Counting misplaced tiles ignores **how far** each tile has to travel. A tile one square from home counts
the same as a tile in the opposite corner.

**Manhattan distance** fixes this: for every tile, add up how many rows and columns it is away from home.

```text
   Tile 6 is here          Tile 6 belongs here

   1  3 [6]                1  2  3
   5  _  2                 4  5 [6]
   4  7  8                 7  8  _

   rows apart    = 1
   columns apart = 0
   distance for tile 6 = 1 + 0 = 1
```

It is called *Manhattan* distance because you may only move along rows and columns, like walking city
blocks — never diagonally.

In [ ]:
def manhattan(board):
    """Add up the row + column distance of every tile from its goal position."""
    total = 0
    for i in range(9):
        tile = board[i]

        if tile == 0:               # skip the blank
            continue

        goal_i = GOAL.index(tile)   # where this tile belongs

        rows_apart = abs(i // 3 - goal_i // 3)
        cols_apart = abs(i % 3 - goal_i % 3)

        total = total + rows_apart + cols_apart
    return total


print("Board     misplaced   manhattan")
print("GOAL         ", misplaced(GOAL), "         ", manhattan(GOAL))
print("EASY         ", misplaced(EASY), "         ", manhattan(EASY))
print("START        ", misplaced(START), "         ", manhattan(START))
print()
print("Manhattan is never smaller than misplaced tiles.")
print("That makes it the BETTER heuristic - it is closer to the truth.")

## 5. Algorithm 1 — Hill Climbing

The simplest informed strategy:

> **Look at every board you can reach in one move. Go to the best one. Repeat.**

It keeps **no list of alternatives** — only the board it is standing on. That makes it very cheap and
very fast, but it has no way to back out of a mistake.

**Where it fails:** if *every* neighbour is worse than the current board, it has nowhere to go and stops —
even though it is not at the goal. That is a **local minimum**.

In [ ]:
def hill_climbing(start, h):
    """Always step to the best neighbour. Stop if no neighbour is better."""
    current = start
    path = [current]

    while current != GOAL:
        neighbours = get_moves(current)

        # the neighbour with the SMALLEST h value
        best = min(neighbours, key=h)

        # if it is not an improvement, we are stuck
        if h(best) >= h(current):
            return path, "STUCK"

        current = best
        path.append(current)

    return path, "SOLVED"


print("===== HILL CLIMBING on EASY =====")
path, result = hill_climbing(EASY, manhattan)
print("Result:", result, "in", len(path) - 1, "moves")
print()
for i, b in enumerate(path):
    print("Step", i, " h =", manhattan(b))
    show(b)
    print()

### Now try the harder board

`EASY` worked. Run the same function on `START` and watch what happens.

In [ ]:
print("===== HILL CLIMBING on START =====")
path, result = hill_climbing(START, manhattan)

print("Result:", result)
print("Boards visited:", len(path))
print()
print("It walked downhill from h =", manhattan(path[0]), "to h =", manhattan(path[-1]), "and then had nowhere to go.")
print()
print("The board it got stuck on:")
show(path[-1])
print()
print("Its neighbours and their h values:")
for b in get_moves(path[-1]):
    print("  h =", manhattan(b), "  <- worse than", manhattan(path[-1]))
print()
print("Every neighbour is worse, so Hill Climbing stops. This is a LOCAL MINIMUM.")

## 6. Keeping alternatives — the frontier

Hill Climbing failed because it threw away every board it did not take. The fix is to **keep a list of
paths we could still follow**. That list is called the **frontier**.

Each item in our frontier is a **path** — a list of boards, starting at `START`.

```text
frontier = [ [START, boardA],
             [START, boardB],
             [START, boardC] ]
```

Each round we **pick one path**, extend it, and put the new paths back. The only difference between
Best First and A\* is **how they pick**:

| Algorithm | Picks the path with the smallest... |
|---|---|
| Best First | `h(last board)` — looks closest to the goal |
| A\* | `moves so far + h(last board)` — best **total** guess |

> The full notebook uses Python's `heapq` for this. Here we simply use `min()`, which is slower but much
> easier to read — and these boards are small enough that it does not matter.

## 7. Algorithm 2 — Best First Search

Pick the path whose **last board has the smallest h**. It never gets permanently stuck, because it always
has other paths to fall back on.

But it ignores how many moves it has already made — so the answer it finds can be **longer than necessary**.

In [ ]:
def best_first(start, h):
    """Always extend the path whose last board LOOKS closest to the goal."""
    frontier = [[start]]        # a list of paths; we begin with one path
    visited = set()
    explored = 0

    while frontier:
        # pick the best-looking path, and take it out of the frontier
        path = min(frontier, key=lambda p: h(p[-1]))
        frontier.remove(path)

        board = path[-1]        # the last board on that path

        if board in visited:
            continue
        visited.add(board)
        explored = explored + 1

        if board == GOAL:
            return path, explored

        # add one new path for every move we could make
        for nb in get_moves(board):
            if nb not in visited:
                frontier.append(path + [nb])

    return None, explored


path, explored = best_first(START, manhattan)
print("===== BEST FIRST SEARCH on START =====")
print("Boards explored:", explored)
print("Moves in the answer:", len(path) - 1)

## 8. Algorithm 3 — A\*

One change: score a path by **what it has already cost plus what it still looks like it will cost**.

```text
   f(path)  =  g          +  h
            =  moves made  +  estimated moves left
            =  len(path)-1 +  h(last board)
```

Because A\* refuses to ignore the moves already made, it cannot be fooled into wandering off — and when
the heuristic never overestimates, **the answer it returns is the shortest one that exists**.

In [ ]:
def a_star(start, h):
    """Like best_first, but the score also counts the moves already made."""
    frontier = [[start]]
    visited = set()
    explored = 0

    while frontier:
        # THE ONLY LINE THAT CHANGED: add the moves already made
        path = min(frontier, key=lambda p: (len(p) - 1) + h(p[-1]))
        frontier.remove(path)

        board = path[-1]

        if board in visited:
            continue
        visited.add(board)
        explored = explored + 1

        if board == GOAL:
            return path, explored

        for nb in get_moves(board):
            if nb not in visited:
                frontier.append(path + [nb])

    return None, explored


path, explored = a_star(START, manhattan)
print("===== A* SEARCH on START =====")
print("Boards explored:", explored)
print("Moves in the answer:", len(path) - 1)
print()
print("The full solution:")
print()
for i, b in enumerate(path):
    print("START" if i == 0 else ("GOAL" if i == len(path) - 1 else "Move " + str(i)))
    show(b)
    print()

## 9. Compare all three

Run the three algorithms on the same board and put the numbers side by side.

In [ ]:
print("Algorithm      Heuristic    Explored   Moves   Comment")
print("-" * 68)

# Hill Climbing
path, result = hill_climbing(START, manhattan)
if result == "STUCK":
    print("Hill Climbing  manhattan    %8d   %5s   got stuck, no answer" % (len(path), "-"))
else:
    print("Hill Climbing  manhattan    %8d   %5d   solved" % (len(path), len(path) - 1))

# Best First
path_bf, n_bf = best_first(START, manhattan)
print("Best First     manhattan    %8d   %5d   found an answer, but NOT the shortest" % (n_bf, len(path_bf) - 1))

# A* with each heuristic
path_a1, n_a1 = a_star(START, misplaced)
print("A*             misplaced    %8d   %5d   shortest answer" % (n_a1, len(path_a1) - 1))

path_a2, n_a2 = a_star(START, manhattan)
print("A*             manhattan    %8d   %5d   shortest answer, fewer boards" % (n_a2, len(path_a2) - 1))

print()
print("Two things to notice:")
print()
print("1. Best First returned", len(path_bf) - 1, "moves but A* returned", len(path_a2) - 1, "moves.")
print("   Best First stops at the FIRST goal it reaches. Because it ignores the moves")
print("   already made, that answer is not the shortest one.")
print()
print("2. A* explored", n_a2, "boards with manhattan but", n_a1, "boards with misplaced.")
print("   Manhattan is the better estimate, so it prunes more and A* does less work.")
print()
print("(Greedy search is often the cheapest to run, but not always - here it chased a")
print(" promising-looking board down a bad path and ended up doing more work than A*.)")

## 10. What each algorithm is for

| | Keeps alternatives? | Uses | Always finds an answer? | Shortest answer? | Memory |
|---|---|---|---|---|---|
| **Hill Climbing** | No | `h` | No — it can get stuck | No | tiny |
| **Best First** | Yes | `h` | Yes | **No** | large |
| **A\*** | Yes | `g + h` | Yes | **Yes**, if h never overestimates | large |

**Use A\*** when you need the best answer and can afford the memory.
**Use Hill Climbing** when the space is far too big to store a frontier and a good-enough answer will do.

### One limitation to remember

A\* stores the whole frontier, so its memory grows with the search. On the 8-puzzle that is fine
(181,440 states). On the 15-puzzle there are about 10<sup>13</sup> states and plain A\* runs out of memory —
which is why variants such as IDA\* exist.

## 11. Your turn

Try these in the empty cell below. Each one is a small change to code you already have.

1. **Change the start board.** Try `(1, 2, 3, 4, 5, 6, 0, 7, 8)` and then a board of your own.
   Does A\* still return the shortest answer?
2. **Break the heuristic.** Write `def silly(board): return 100` and run A\* with it.
   It *overestimates*, so it is not admissible — is the answer still the shortest?
3. **Count the difference.** Run A\* with `misplaced` and with `manhattan` on three different boards and
   note how many boards each explores.
4. **Rescue Hill Climbing.** If `hill_climbing` returns `"STUCK"`, run it again from a *random* neighbour
   and keep the best result. That is the **random restart** idea from the lecture.

In [ ]:
# Your practice area - write your own experiments here

## 12. Summary

- A **heuristic** `h` estimates how far a board is from the goal. `h = 0` means we are there.
- **Misplaced tiles** is the simplest heuristic; **Manhattan distance** is better, because it also counts
  *how far* each tile must travel.
- A heuristic is **admissible** if it never overestimates. Admissibility is what makes A\* optimal.
- **Hill Climbing** keeps no alternatives, so it is cheap and can get **stuck at a local minimum**.
- **Best First** keeps alternatives and picks by `h` alone — fast, but the answer may be longer than needed.
- **A\*** picks by `g + h` — moves already made plus moves still estimated. With an admissible heuristic it
  returns the **shortest possible answer**.

> The single sentence to leave with:
> **Uninformed search uses only g. Best First uses only h. A\* uses both — and that is why it wins.**